In [28]:
suppressPackageStartupMessages({
    library(data.table)
    library(SingleCellExperiment)
    library(dplyr)
    library(Matrix)
    library(matrixStats)
})

Check how original atlas looks

In [7]:
original = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation/pijuansala2019_gastrulation10x/'
original_meta = fread(paste0(original, 'sample_metadata.txt.gz'))
original_sce = readRDS(paste0(original, 'processed/SingleCellExperiment.rds'))

Warning message in fread(paste0(original, "sample_metadata.txt.gz")):
“Previous fread() session was not cleaned up properly. Cleaned up ok at the beginning of this fread() call.”


In [8]:
head(original_meta)

cell,barcode,sample,stage,sequencing.batch,doublet,stripped,celltype,umapX,umapY,nFeature_RNA,nCount_RNA
<chr>,<chr>,<int>,<chr>,<int>,<lgl>,<lgl>,<chr>,<dbl>,<dbl>,<int>,<int>
cell_1,AAAGGCCTCCACAA,1,E6.5,1,FALSE,FALSE,Epiblast,-10.2275459,-2.8816875,2547,8963
cell_10,AACTGTCTTCGCAA,1,E6.5,1,FALSE,FALSE,Epiblast,-11.2435319,-0.8761099,1933,5643
cell_100,CACAGATGGGGACA,1,E6.5,1,FALSE,FALSE,Epiblast,-10.9044295,-0.9639773,4278,24947
cell_1000,GCCACTACCCGCTT,3,E7.5,1,FALSE,FALSE,Caudal_epiblast,-2.9437001,-0.7820226,2829,9294
cell_10000,GTAGGTACGTGTTG,8,E7.75,1,FALSE,FALSE,Rostral_neurectoderm,-7.3668170,-2.2819553,4492,24351
cell_100000,GACTGATGACACAC,29,E8.5,3,FALSE,FALSE,Forebrain_Midbrain_Hindbrain,-0.9185215,-5.5175930,3217,14074


In [9]:
original_sce

class: SingleCellExperiment 
dim: 29452 116312 
metadata(1): log.exprs.offset
assays(1): counts
rownames(29452): ENSMUSG00000051951 ENSMUSG00000089699 ...
  ENSMUSG00000096730 ENSMUSG00000095742
rowData names(0):
colnames(116312): cell_1 cell_2 ... cell_139330 cell_139331
colData names(0):
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):

Create same files for new atlas

In [1]:
new = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/'
dir.create(paste0(new, 'processed'))

Warning message in dir.create(paste0(new, "processed")):
“'/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/processed' already exists”


In [3]:
count = readMM(paste0(new, 'raw_counts.mtx'))
genes = fread(paste0(new, 'genes.tsv'), header=FALSE)
meta = fread(paste0(new, 'meta.tab'))

In [4]:
count[1:10, 1:10]

10 x 10 sparse Matrix of class "dgTMatrix"
                         
 [1,] 1 . . . . . . . . .
 [2,] . . . . . 1 . . 1 .
 [3,] 1 9 8 4 5 8 1 4 1 4
 [4,] . . . . . . . . . .
 [5,] . . . . . . . . . .
 [6,] . . . . . . . . . .
 [7,] 4 3 1 1 1 2 . 2 1 2
 [8,] . . 1 . . 4 . 1 . .
 [9,] . 1 1 . 1 . . . . .
[10,] . . . 2 1 . 2 . 1 1

In [5]:
dim(count)

[1]  23972 115559

In [6]:
rownames(count) = genes$V1
colnames(count) = meta$cell

In [7]:
meta$doublet = FALSE
meta$stripped = FALSE
meta$idx = 1:nrow(meta)

In [8]:
umap = fread(paste0(new, 'umap.csv'))

In [9]:
meta = merge(meta, umap, by= 'cell') %>% .[order(idx)]

In [10]:
sizefactors = fread(paste0(new, 'sizefactors.tab'))

In [11]:
sce = SingleCellExperiment(list(counts = count))

In [12]:
sce

class: SingleCellExperiment 
dim: 23972 115559 
metadata(0):
assays(1): counts
rownames(23972): ENSMUSG00000001138 ENSMUSG00000001143 ...
  ENSMUSG00000108929 ENSMUSG00000109022
rowData names(0):
colnames(115559): cell_1 cell_2 ... ext_cell_334312 ext_cell_311352
colData names(0):
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):

In [14]:
sizeFactors(sce) = sizefactors$V1 

In [ ]:
saveRDS(sce, paste0(new, 'processed/SingleCellExperiment.rds'))
fwrite(meta, paste0(new, 'sample_metadata.txt.gz'))

In [30]:
gene_stats = data.frame(ens_id = rownames(sce), var_pseudobulk = rowVars(as.matrix(counts(sce))))

In [37]:
gene_meta = fread("/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/Mmusculus_genes_BioMart.87.txt")[,c('ens_id', 'symbol')] 
colnames(gene_meta) = c('ens_id', 'gene')

In [39]:
gene_stats = merge(gene_stats, gene_meta)

In [43]:
dir.create(paste0(new, '/results/gene_statistics/'), recursive = TRUE)

In [44]:
fwrite(gene_stats, paste0(new, '/results/gene_statistics/gene_statistics.txt.gz'))

In [16]:
saveRDS(sce, paste0(new, 'processed/SingleCellExperiment.rds'))
fwrite(meta, paste0(new, 'sample_metadata.txt.gz'))

In [20]:
unique(meta$stage)

[1] "E6.5"  "E7.5"  "E6.75" "E7.75" "E7.0"  "E8.0"  "E8.5"  "E7.25" "E8.25"
[10] "E9.0"  "E8.75" "E9.25" "E9.5"

In [18]:
unique(meta$celltype)

[1] "Epiblast"                                      
 [2] "Primitive Streak"                              
 [3] "ExE ectoderm"                                  
 [4] "Visceral endoderm"                             
 [5] "ExE endoderm"                                  
 [6] "Non-neural ectoderm"                           
 [7] "Nascent mesoderm"                              
 [8] "Parietal endoderm"                             
 [9] "Ectoderm"                                      
[10] "Anterior Primitive Streak"                     
[11] "Haematoendothelial progenitors"                
[12] "Caudal epiblast"                               
[13] "Blood progenitors"                             
[14] "Intermediate mesoderm"                         
[15] "Paraxial mesoderm"                             
[16] "Lateral plate mesoderm"                        
[17] "Mesenchyme"                                    
[18] "PGC"                                           
[19] "Node"                                          
[20] "Gut tube"                                      
[21] "Embryo proper endothelium"                     
[22] "Cardiopharyngeal progenitors SHF"              
[23] "Notochord"                                     
[24] "Amniotic ectoderm"                             
[25] "Venous endothelium"                            
[26] "Presomitic mesoderm"                           
[27] "Cardiomyocytes FHF 1"                          
[28] "Allantois"                                     
[29] "Cranial mesoderm"                              
[30] "EMP"                                           
[31] "Limb mesoderm"                                 
[32] "Anterior somitic tissues"                      
[33] "Pharyngeal mesoderm"                           
[34] "Allantois endothelium"                         
[35] "Thyroid primordium"                            
[36] "Erythroid"                                     
[37] "Hindbrain neural progenitors"                  
[38] "Cardiomyocytes SHF 1"                          
[39] "NMPs"                                          
[40] "Pharyngeal endoderm"                           
[41] "Dorsal spinal cord progenitors"                
[42] "Anterior cardiopharyngeal progenitors"         
[43] "Placodal ectoderm"                             
[44] "Optic vesicle"                                 
[45] "Ventral forebrain progenitors"                 
[46] "Spinal cord progenitors"                       
[47] "Hindgut"                                       
[48] "Caudal mesoderm"                               
[49] "Embryo proper mesothelium"                     
[50] "Neural tube"                                   
[51] "Midbrain/Hindbrain boundary"                   
[52] "Posterior somitic tissues"                     
[53] "Midgut"                                        
[54] "Migratory neural crest"                        
[55] "Ventral hindbrain progenitors"                 
[56] "Surface ectoderm"                              
[57] "YS mesothelium"                                
[58] "Limb ectoderm"                                 
[59] "Somitic mesoderm"                              
[60] "NMPs/Mesoderm-biased"                          
[61] "Cardiomyocytes FHF 2"                          
[62] "Foregut"                                       
[63] "Dermomyotome"                                  
[64] "Kidney primordium"                             
[65] "Otic placode"                                  
[66] "Cardiomyocytes SHF 2"                          
[67] "Midbrain progenitors"                          
[68] "Cardiopharyngeal progenitors FHF"              
[69] "Epicardium"                                    
[70] "Hindbrain floor plate"                         
[71] "Late dorsal forebrain progenitors"             
[72] "Dorsal hindbrain progenitors"                  
[73] "Sclerotome"                                    
[74] "YS endothelium"                                
[75] 